In [2]:
%matplotlib inline

In [3]:
import QUANTAXIS as QA
import pandas as pd
import tushare as ts
import numpy as np
import talib as ta
import scipy.signal as signal
import matplotlib.pyplot as plt
import pandasql as sqldf

# 主要指数

In [4]:
index_code_name_map = pd.DataFrame(['上证指数','深证综指','上证50','中证500','创业板指','科创50','深证成指','沪深300'],\
                                   index = ['000001','399106','000016','000905','399006','000688','399001','399300'],columns=['名称'])

In [7]:
start = '2026-01-01'
end = '2026-07-24'

In [8]:
# 获取主要指数数据
main_index_data = QA.QA_fetch_index_day_adv(['000001','399106','000016','000905','399006','000688','399001','399300'],start,end)

# 行业分类

In [9]:
# 读取申万二级行业信息
sw2_industry = pd.read_csv('申万细分行业.csv')
sw2_industry_name_code_map = sw2_industry.filter(['名称','blk_code']).set_index('名称')
sw2_industry.loc[:,('blk_code')] = sw2_industry['代码'].astype(str)

In [10]:
sw2_industry

,代码,名称,blk_code
0,880313,石油贸易,880313
1,880363,林业,880363
2,880431,船舶,880431
3,880490,通信设备,880490
4,880444,农用机械,880444
...,...,...,...
105,880302,煤炭开采,880302
106,880373,乳制品,880373
107,880351,矿物制品,880351
108,880382,啤酒,880382


In [11]:
all_stock_basic_snap = pd.read_csv(r"D:\zd_gxzq\T0002\export\20260711.csv",encoding='utf-8-sig')[:-1]

In [13]:
# 把数字变字符，并左填充0
all_stock_basic_snap['代码']=all_stock_basic_snap['代码'].astype(str).str.zfill(6)

In [18]:
# 国信证券数据
# 清洗数据 拿掉亿
all_stock_basic_snap.loc[:,'ab_mv_snap'] = all_stock_basic_snap['总市值'].map(lambda x: x.strip().replace('亿',''))
all_stock_basic_snap.loc[:,'circulate_mv_snap'] = all_stock_basic_snap['流通市值'].map(lambda x: x.strip().replace('亿',''))
# 排除未上市
all_stock_basic = all_stock_basic_snap[all_stock_basic_snap['ab_mv_snap'] != '--']
#字符转数字
all_stock_basic.loc[:,'ab_mv'] = all_stock_basic['ab_mv_snap'].map(lambda x: pd.to_numeric(x))
all_stock_basic.loc[:,'circulate_mv'] = all_stock_basic['circulate_mv_snap'].map(lambda x: pd.to_numeric(x))

# 聚合细分行业市值
blk_mv_sum = pd.DataFrame(all_stock_basic.groupby(['细分行业'])['circulate_mv'].agg(['sum','count'])).rename(columns={'sum':'blk_mv','count':'blk_cnt'})  
# # columns={'sum':'blk_mv','count':'blk_cnt'}
sw2_industry_name_code_map = sw2_industry.filter(['名称','blk_code']).set_index('名称')

# 合并板块指数代码
blk_info = pd.concat([blk_mv_sum,sw2_industry_name_code_map],axis=1)

<ipython-input-18-45a1c90e5feb>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_stock_basic.loc[:,'ab_mv'] = all_stock_basic['ab_mv_snap'].map(lambda x: pd.to_numeric(x))
<ipython-input-18-45a1c90e5feb>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_stock_basic.loc[:,'circulate_mv'] = all_stock_basic['circulate_mv_snap'].map(lambda x: pd.to_numeric(x))


In [21]:
blk_info['blk_mv_rank'] = blk_info['blk_mv'].rank(ascending=False)

In [23]:
condition = [
        (blk_info['blk_mv_rank']<=5), #超大盘
        (blk_info['blk_mv_rank']<=20), # 大盘
        (blk_info['blk_mv_rank']<=40),# 中盘
        (blk_info['blk_mv_rank']<=65), # 中小
        (blk_info['blk_mv_rank']<=90), #小型
        (blk_info['blk_mv_rank']>90) # 迷你
]
choices = ['超大', '大盘', '中盘','中小','小型','迷你']

In [24]:
# 对板块市值进行分类
blk_basics = blk_mv_sum.join(sw2_industry.drop('代码',axis=1).set_index('名称')) 
blk_info['category'] = np.select(condition, choices, default=np.nan)

In [172]:
blk_info.index.name = 'blk_name'

In [173]:
blk_info.sort_values(by='blk_mv_rank',ascending = True)

,blk_mv,blk_cnt,blk_code,blk_mv_rank,category
blk_name,,,,,
半导体,108068.64,195,880491,1.0,超大
银行,95638.41,42,880471,2.0,超大
元器件,74164.21,309,880492,3.0,超大
通信设备,61028.19,137,880490,4.0,超大
电气设备,59669.78,347,880446,5.0,超大
...,...,...,...,...,...
批发业,217.83,5,880413,106.0,迷你
林业,180.65,4,880363,107.0,迷你
渔业,177.88,7,880362,108.0,迷你


In [31]:
blk_info[blk_info['category']=='大盘']

,blk_mv,blk_cnt,blk_code,blk_mv_rank,category
IT设备,14533.60,81,880489,19.0,大盘
专用机械,30502.13,291,880445,7.0,大盘
保险,18913.81,5,880473,14.0,大盘
化学制药,21234.83,148,880401,12.0,大盘
化工原料,29072.27,257,880336,8.0,大盘
医疗保健,14778.74,184,880398,18.0,大盘
家用电器,17427.26,92,880387,16.0,大盘
小金属,19328.85,67,880329,13.0,大盘
建筑工程,13162.99,128,880477,20.0,大盘
汽车配件,21323.43,266,880392,11.0,大盘


# 分析领头指数

找出和市场走势相关性最强的指数

In [102]:
sw2_industry_code = sw2_industry['blk_code'].to_list()

In [103]:
#获取板块数据
sw2_industry_data = QA.QA_fetch_index_day_adv(sw2_industry_code,start,end)

In [104]:
sw2_industry_idx_corr = pd.concat([sw2_industry_data.pivot('close').pct_change(),main_index_data.pivot('close').pct_change()],axis=1)\
                            .corr()

In [116]:
major_index_list = index_code_name_map.index.to_list()

In [203]:
corr_rate = 0.7
result_df = pd.DataFrame(data=[],columns=['index', 'mv_sum'])
for index_code in index_code_name_map.index.to_list():
#     过滤指定 指数
    sw2_blk_temp = sw2_industry_idx_corr[index_code].filter(regex=r'^880',axis=0)
#     过滤大于相关性参数
    one_blk_related_blk = sw2_blk_temp[sw2_blk_temp>corr_rate].sort_values(ascending= False)  
    main_index_raw = one_blk_related_blk.to_frame().join(blk_info_code_idx)
    mv_sum = main_index_raw['blk_mv'].sum()
    new_row = pd.DataFrame([{'index':index_code,'mv_sum':mv_sum}])
    result_df = pd.concat([result_df, new_row], ignore_index=True)
#     带入大盘指数名
blk_info_code_idx = blk_info.reset_index().set_index('blk_code')
result_df = result_df.merge(index_code_name_map,left_on='index', right_index=True, how='left')

In [206]:
result_df.sort_values(by='mv_sum',ascending = False)

,index,mv_sum,名称
3,000905,459255.26,中证500
1,399106,454772.34,深证综指
6,399001,400117.52,深证成指
4,399006,356243.91,创业板指
7,399300,351944.12,沪深300
5,000688,288296.77,科创50
0,000001,245097.54,上证指数
2,000016,31198.02,上证50


In [211]:
# 总市值
blk_info['blk_mv'].sum()

1021835.9800000002

# 各个板块和对应关联性高的指数

In [291]:
blk_result_df = pd.DataFrame(data=[],columns=['blk_code','index', 'mv_corr'])
for blk_code in sw2_industry.blk_code.to_list():
    s = sw2_industry_idx_corr[blk_code]
    filtered_s = s[~s.index.str.startswith('880') & (s.index != blk_code)]
    max_idx = filtered_s.idxmax()
    max_val = filtered_s[max_idx]
    max_corr = filtered_s.max()
    new_row = pd.DataFrame([{'blk_code':blk_code,'index':max_idx,'mv_corr': max_val}])
    blk_result_df = pd.concat([blk_result_df,new_row], ignore_index=True)

# 结合blk_info    
blk_info_code_idx = blk_info.reset_index().set_index('blk_code')
blk_result_df = blk_result_df.merge(blk_info_code_idx,left_on='blk_code', right_index=True, how='left')

In [292]:
blk_result_df.sort_values(by = 'blk_mv_rank')

,blk_code,index,mv_corr,blk_name,blk_mv,blk_cnt,blk_mv_rank,category
18,880491,000688,0.976101,半导体,108068.64,195,1.0,超大
78,880471,000016,0.104014,银行,95638.41,42,2.0,超大
7,880492,399001,0.896615,元器件,74164.21,309,3.0,超大
3,880490,399006,0.877580,通信设备,61028.19,137,4.0,超大
85,880446,399106,0.810877,电气设备,59669.78,347,5.0,超大
...,...,...,...,...,...,...,...,...
69,880413,000001,0.559972,批发业,217.83,5,106.0,迷你
1,880363,399106,0.171658,林业,180.65,4,107.0,迷你
63,880362,000001,0.443862,渔业,177.88,7,108.0,迷你
21,880463,399106,0.372255,公路,166.65,3,109.0,迷你


# 板块和成分股的相关性

In [296]:
all_blk_component = all_stock_snap_blk.filter(['代码','细分行业','blk_code','circulate_share','blk_mv_rank'])\
        .groupby(['细分行业','blk_code','blk_mv_rank']).agg(list)['代码']

In [302]:
all_blk_component['半导体'].values[0]

['001270',
 '001309',
 '002049',
 '002077',
 '002119',
 '002156',
 '002185',
 '002213',
 '002371',
 '002409',
 '002449',
 '003026',
 '003043',
 '300046',
 '300053',
 '300077',
 '300102',
 '300123',
 '300223',
 '300236',
 '300241',
 '300301',
 '300303',
 '300323',
 '300327',
 '300346',
 '300373',
 '300456',
 '300458',
 '300604',
 '300613',
 '300623',
 '300632',
 '300655',
 '300661',
 '300666',
 '300671',
 '300672',
 '300708',
 '300782',
 '300831',
 '301095',
 '301269',
 '301297',
 '301308',
 '301348',
 '301369',
 '301536',
 '301581',
 '301611',
 '301629',
 '301678',
 '600171',
 '600206',
 '600360',
 '600460',
 '600520',
 '600584',
 '600641',
 '600667',
 '600703',
 '600745',
 '600877',
 '603005',
 '603061',
 '603068',
 '603078',
 '603160',
 '603290',
 '603375',
 '603501',
 '603690',
 '603893',
 '603986',
 '603991',
 '605111',
 '605358',
 '688008',
 '688012',
 '688018',
 '688019',
 '688035',
 '688037',
 '688041',
 '688045',
 '688047',
 '688048',
 '688049',
 '688052',
 '688061',
 '688072',

In [304]:
start_date = '2026-05-01'
end_date = '2026-07-10'

In [293]:
blk_data = QA.QA_fetch_index_day_adv(['880491'],start,end)

In [307]:
QA.QA_fetch_index_day_adv(['880402'],start,end).data

,,open,close,high,low,vol,amount,up_count,down_count,date_stamp,volume
date,code,,,,,,,,,,
2026-01-05,880402,2461.77,2542.52,2546.35,2450.59,83951.0,2.126726e+10,74,5,1.767542e+09,83951.0
2026-01-06,880402,2544.63,2550.24,2551.01,2526.15,72082.0,1.681782e+10,50,26,1.767629e+09,72082.0
2026-01-07,880402,2548.10,2590.68,2595.40,2548.10,69788.0,2.049374e+10,49,30,1.767715e+09,69788.0
2026-01-08,880402,2592.53,2596.07,2607.86,2586.38,71767.0,1.925947e+10,55,25,1.767802e+09,71767.0
2026-01-09,880402,2595.30,2645.83,2647.65,2583.21,103046.0,2.594339e+10,69,9,1.767888e+09,103046.0
...,...,...,...,...,...,...,...,...,...,...,...
2026-07-06,880402,2278.65,2320.84,2413.73,2266.32,147940.0,4.124545e+10,58,20,1.783267e+09,147940.0
2026-07-07,880402,2290.25,2205.57,2292.02,2204.68,110172.0,3.024617e+10,3,76,1.783354e+09,110172.0
2026-07-08,880402,2212.84,2164.57,2243.06,2164.57,103771.0,2.449378e+10,22,56,1.783440e+09,103771.0


In [305]:
print(f"QUANTAXIS版本: {QA.__version__}")

QUANTAXIS版本: 1.10.19


In [295]:
blk_data.data

,,open,close,high,low,vol,amount,up_count,down_count,date_stamp,volume
date,code,,,,,,,,,,
2026-01-05,880491,4136.87,4248.86,4248.86,4136.87,364512.0,2.227130e+11,180,11,1.767542e+09,364512.0
2026-01-06,880491,4241.79,4331.13,4373.73,4238.17,395774.0,2.384924e+11,150,40,1.767629e+09,395774.0
2026-01-07,880491,4406.48,4430.52,4459.32,4373.87,469473.0,2.907078e+11,141,46,1.767715e+09,469473.0
2026-01-08,880491,4419.92,4473.80,4544.45,4419.92,425097.0,2.637204e+11,113,76,1.767802e+09,425097.0
2026-01-09,880491,4433.71,4516.55,4521.39,4398.55,441903.0,2.524089e+11,126,62,1.767888e+09,441903.0
...,...,...,...,...,...,...,...,...,...,...,...
2026-07-06,880491,7444.55,7316.84,7484.75,7021.28,639412.0,5.608009e+11,74,120,1.783267e+09,639412.0
2026-07-07,880491,7185.94,7337.87,7483.10,7126.94,593139.0,4.953981e+11,103,91,1.783354e+09,593139.0
2026-07-08,880491,7406.40,7323.09,7592.43,7083.81,653520.0,5.352022e+11,73,121,1.783440e+09,653520.0


# 获取行业龙头

In [33]:
all_stock_snap_blk = all_stock_basic.merge(blk_info,left_on = '细分行业',right_index=True)

In [39]:
# 获取
all_stock_snap_blk.loc[:,'mv_share'] = all_stock_snap_blk['ab_mv']/all_stock_snap_blk['blk_mv']*100
all_stock_snap_blk.loc[:,'circulate_share'] = all_stock_snap_blk['circulate_mv']/all_stock_snap_blk['blk_mv']*100
# 对个股市值进行排序
all_stock_snap_blk.loc[:,'circulate_share_rk'] = all_stock_snap_blk.groupby(['细分行业'])['circulate_share']\
                                            .rank(ascending=0,method='min')  ## dense
all_stock_snap_blk.loc[:,'mv_share_rk'] = all_stock_snap_blk.groupby(['细分行业'])['circulate_share']\
                                            .rank(ascending=0,method='min')  ## dense

In [41]:
# 龙头
all_stock_snap_blk.query("circulate_share_rk==1").filter(["名称","细分行业","circulate_share","blk_mv", 'blk_mv_rank'])\
        .query('blk_mv_rank<=20')\
        .sort_values(by='blk_mv',ascending = False)

,名称,细分行业,circulate_share,blk_mv,blk_mv_rank
4811,寒武纪,半导体,8.139364,108068.64,1.0
3727,农业银行,银行,20.795944,95638.41,2.0
976,立讯精密,元器件,6.137462,74164.21,3.0
3694,工业富联,通信设备,21.548534,61028.19,4.0
2190,宁德时代,电气设备,24.881674,59669.78,5.0
2915,中信证券,证券,11.084485,31198.02,6.0
404,华工科技,专用机械,5.208095,30502.13,7.0
3120,万华化学,化工原料,7.398631,29072.27,8.0
1895,润泽科技,软件服务,5.391849,24959.90,9.0
3268,贵州茅台,白酒,66.142572,22773.88,10.0


In [43]:
# 查看板块前10，每个板块前3的个股
all_stock_snap_blk.query("circulate_share_rk<=3")\
.filter(["名称","细分行业","circulate_share","blk_mv_rank"])\
.sort_values(by=['blk_mv_rank','circulate_share'],ascending = [True,False])\
.query('blk_mv_rank <=10')

,名称,细分行业,circulate_share,blk_mv_rank
4811,寒武纪,半导体,8.139364,1.0
4626,海光信息,半导体,7.592314,1.0
880,北方华创,半导体,5.374158,1.0
3727,农业银行,银行,20.795944,2.0
3746,工商银行,银行,20.663853,2.0
3855,XD中国银,银行,12.759852,2.0
976,立讯精密,元器件,6.137462,3.0
3028,生益科技,元器件,4.823283,3.0
893,东山精密,元器件,4.529220,3.0
3694,工业富联,通信设备,21.548534,4.0


In [44]:
# 查看板块前10，每个板块前3的个股
all_stock_snap_blk.query("mv_share_rk<=3")\
.filter(["名称","细分行业","mv_share","blk_mv_rank"])\
.sort_values(by=['blk_mv_rank','mv_share'],ascending = [True,False])\
.query('blk_mv_rank <=10')

,名称,细分行业,mv_share,blk_mv_rank
4811,寒武纪,半导体,8.139364,1.0
4626,海光信息,半导体,7.592314,1.0
880,北方华创,半导体,5.378776,1.0
3746,工商银行,银行,27.315992,2.0
3727,农业银行,银行,22.798309,2.0
3855,XD中国银,银行,19.506912,2.0
976,立讯精密,元器件,6.472785,3.0
893,东山精密,元器件,5.983991,3.0
3028,生益科技,元器件,4.893007,3.0
3694,工业富联,通信设备,21.548534,4.0


# 获取垄断龙头

In [46]:
# 个股占比最高的前50股
all_stock_snap_blk.query("circulate_share_rk==1 and blk_mv_rank<=50")\
    .filter(['代码','名称','细分行业','circulate_mv','circulate_share'])\
    .sort_values(by='circulate_share',ascending=False)

,代码,名称,细分行业,circulate_mv,circulate_share
3815,601857,中国石油,石油开采,15010.18,81.497472
2913,600028,中国石化,石油加工,4510.22,72.317064
3268,600519,贵州茅台,白酒,15063.23,66.142572
3000,600150,中国船舶,船舶,2787.49,58.884455
3797,601728,中国电信,电信运营,4378.32,55.100931
3578,600900,长江电力,水力发电,6858.44,53.569175
3832,601899,紫金矿业,铜,5710.82,52.231024
3671,601088,中国神华,煤炭开采,6932.83,44.938887
3776,601628,中国人寿,保险,7792.16,41.198257
3838,601919,中远海控,水运,1755.40,33.847453


# 获取板块成分

In [75]:
all_blk_component = all_stock_snap_blk.filter(['代码','细分行业','blk_code','circulate_share','blk_mv_rank'])\
        .groupby(['细分行业','blk_code','blk_mv_rank']).agg(list)['代码']
all_blk_component

细分行业  blk_code  blk_mv_rank
IT设备  880489    19.0           [000066, 000938, 000977, 000997, 001229, 00133...
专用机械  880445    7.0            [000519, 000551, 000856, 000880, 000988, 00122...
中成药   880403    36.0           [000423, 000538, 000590, 000623, 000650, 00079...
乳制品   880373    65.0           [001318, 002329, 002570, 002719, 002732, 00291...
互联网   880494    27.0           [000676, 000681, 002095, 002115, 002123, 00212...
                                                     ...                        
银行    880471    2.0            [000001, 001227, 002142, 002807, 002839, 00293...
陶瓷    880345    93.0                    [002162, 002918, 003012, 300234, 300285]
食品    880375    29.0           [000505, 000523, 000529, 000639, 000716, 00089...
饲料    880364    67.0           [000702, 000876, 001313, 001366, 002100, 00231...
黄金    880328    49.0           [000506, 000975, 001337, 002155, 002237, 30013...
Name: 代码, Length: 110, dtype: object

In [81]:
all_blk_component['半导体'].values  #[0][0]

array([list(['001270', '001309', '002049', '002077', '002119', '002156', '002185', '002213', '002371', '002409', '002449', '003026', '003043', '300046', '300053', '300077', '300102', '300123', '300223', '300236', '300241', '300301', '300303', '300323', '300327', '300346', '300373', '300456', '300458', '300604', '300613', '300623', '300632', '300655', '300661', '300666', '300671', '300672', '300708', '300782', '300831', '301095', '301269', '301297', '301308', '301348', '301369', '301536', '301581', '301611', '301629', '301678', '600171', '600206', '600360', '600460', '600520', '600584', '600641', '600667', '600703', '600745', '600877', '603005', '603061', '603068', '603078', '603160', '603290', '603375', '603501', '603690', '603893', '603986', '603991', '605111', '605358', '688008', '688012', '688018', '688019', '688035', '688037', '688041', '688045', '688047', '688048', '688049', '688052', '688061', '688072', '688082', '688099', '688107', '688110', '688120', '688123', '688126', '688130

In [10]:
QA.QA_fetch_index_day_adv('880347',start,end).data

,,open,close,high,low,vol,amount,up_count,down_count,date_stamp,volume
date,code,,,,,,,,,,
2026-01-05,880347,2261.56,2291.36,2304.95,2255.65,91141.0,1.504546e+10,16,2,1.767542e+09,91141.0
2026-01-06,880347,2290.00,2289.26,2303.75,2266.70,91651.0,1.504907e+10,13,5,1.767629e+09,91651.0
2026-01-07,880347,2272.26,2280.54,2293.36,2241.81,82460.0,1.462801e+10,7,11,1.767715e+09,82460.0
2026-01-08,880347,2263.57,2279.59,2288.94,2253.21,81538.0,1.384567e+10,13,4,1.767802e+09,81538.0
2026-01-09,880347,2275.04,2290.82,2300.54,2248.20,87620.0,1.460410e+10,7,10,1.767888e+09,87620.0
...,...,...,...,...,...,...,...,...,...,...,...
2026-07-20,880347,4067.19,3562.48,4090.34,3533.31,99338.0,2.669838e+10,1,18,1.784477e+09,99338.0
2026-07-21,880347,3652.49,3855.80,3866.45,3291.15,115301.0,3.016500e+10,16,2,1.784563e+09,115301.0
2026-07-22,880347,3825.14,3646.74,3950.50,3626.87,98820.0,2.783846e+10,3,15,1.784650e+09,98820.0
